# AQUA20 — Marine Species Classification

Two-stage transfer learning on AQUA20 (20-class marine species).

Set `MODEL_NAME` below to one of: `resnet50` | `convnext` | `swin`

In [ ]:
!pip install -q "torch>=2.0.0" "torchvision>=0.15.0" "datasets>=2.14.0" "scikit-learn>=1.2.0" "matplotlib>=3.7.0" "seaborn>=0.12.0" "tqdm>=4.65.0" "numpy>=1.24.0" "Pillow>=9.5.0" "opencv-python-headless>=4.7.0"

In [ ]:
import os, json
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.models import (
    ResNet50_Weights, ConvNeXt_Tiny_Weights, Swin_V2_B_Weights,
)
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datasets import load_dataset, load_dataset_builder
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

matplotlib.use("Agg")

## Configuration

In [ ]:
# ── Choose architecture: resnet50 | convnext | swin ──
MODEL_NAME = "resnet50"

NUM_CLASSES = 20
BATCH_SIZE = 32
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 75
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# /kaggle/working IS the kernel's output directory — do not copy into a
# subfolder of it, that just ships everything twice.
#
# `kaggle kernels output` downloads alphabetically and truncates on large
# payloads, so names are chosen to put the small, important artifacts FIRST
# and the 94 MB checkpoint LAST:
#     analysis/  <  confusion_matrix_*  <  gradcam/  <  training_state_*  <  weights_*
# A truncated pull then costs you the weights, never the numbers.
ANALYSIS_DIR = "analysis"
GRADCAM_DIR = f"gradcam/{MODEL_NAME}"
CONFUSION_PATH = f"confusion_matrix_{MODEL_NAME}.png"
STATE_PATH = f"training_state_{MODEL_NAME}.json"
CHECKPOINT_PATH = f"weights_{MODEL_NAME}.pth"

NUM_WORKERS = os.cpu_count()

os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(GRADCAM_DIR, exist_ok=True)
print(f"Device: {DEVICE} | Model: {MODEL_NAME}")
print(f"Num workers: {NUM_WORKERS}")

## Data Module — Load AQUA20 with WeightedRandomSampler

In [ ]:
def get_transform(train=True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(IMG_SIZE + 32),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

def transform_images(examples, transform_fn):
    examples["image"] = [transform_fn(img.convert("RGB")) for img in examples["image"]]
    return examples

def get_dataloaders(use_weighted_sampler=True):
    dataset = load_dataset("taufiktrf/AQUA20")
    split = dataset["train"].train_test_split(test_size=0.2, seed=42)
    train_split, val_split = split["train"], split["test"]
    test_split = dataset["test"]

    if use_weighted_sampler:
        train_labels = train_split["label"]
        class_counts = Counter(train_labels)
        sample_weights = [1.0 / class_counts[l] for l in train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        shuffle = False
    else:
        sampler = None
        shuffle = True

    train_split.set_transform(lambda ex: transform_images(ex, get_transform(train=True)))
    val_split.set_transform(lambda ex: transform_images(ex, get_transform(train=False)))
    test_split.set_transform(lambda ex: transform_images(ex, get_transform(train=False)))

    train_loader = DataLoader(train_split, batch_size=BATCH_SIZE, sampler=sampler,
                              shuffle=shuffle, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(use_weighted_sampler=True)
print("Data module ready.")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## Model Builder — Supports ResNet50, ConvNeXt, SwinV2

In [ ]:
def build_resnet50(num_classes=NUM_CLASSES):
    model = torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    nn.init.kaiming_normal_(model.fc.weight, mode="fan_out", nonlinearity="relu")
    nn.init.zeros_(model.fc.bias)
    return model, "fc"

def build_convnext(num_classes=NUM_CLASSES):
    model = torchvision.models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    nn.init.kaiming_normal_(model.classifier[-1].weight, mode="fan_out", nonlinearity="relu")
    nn.init.zeros_(model.classifier[-1].bias)
    return model, "classifier"

def build_swin(num_classes=NUM_CLASSES):
    model = torchvision.models.swin_v2_b(weights=Swin_V2_B_Weights.IMAGENET1K_V1)
    model.head = nn.Linear(model.head.in_features, num_classes)
    nn.init.kaiming_normal_(model.head.weight, mode="fan_out", nonlinearity="relu")
    nn.init.zeros_(model.head.bias)
    return model, "head"

BUILDERS = {
    "resnet50": build_resnet50,
    "convnext": build_convnext,
    "swin": build_swin,
}

# Grad-CAM target layer per architecture, plus whether that layer emits
# (N, H, W, C) instead of (N, C, H, W). Swin's only Conv2d is the patch-embed
# stem, so searching for the "last conv" would target its very first layer.
GRADCAM_TARGETS = {
    "resnet50": (lambda m: m.layer4[-1], False),
    "convnext": (lambda m: m.features[-1], False),
    "swin": (lambda m: m.features[-1], True),
}

def freeze_backbone(model, classifier_attr):
    for param in model.parameters():
        param.requires_grad = False
    for param in getattr(model, classifier_attr).parameters():
        param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

model, classifier_attr = BUILDERS[MODEL_NAME]()
model = model.to(DEVICE)
print(f"{MODEL_NAME} built. Params: {sum(p.numel() for p in model.parameters()):,}")

## Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, desc):
    model.train()
    total_loss = correct = total = 0
    pbar = tqdm(loader, desc=desc)
    for batch in pbar:
        images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"loss": f"{total_loss/(pbar.n+1):.4f}", "acc": f"{correct/total:.4f}"})
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for batch in loader:
            images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

## Training — Two-Stage Transfer Learning

In [ ]:
criterion = nn.CrossEntropyLoss()
resume = os.path.exists(CHECKPOINT_PATH)
best_acc = 0.0

if resume:
    print("Checkpoint found — loading model, skipping Stage 1")
    model, _ = BUILDERS[MODEL_NAME]()
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
    model = model.to(DEVICE)
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            best_acc = json.load(f).get("best_val_acc", 0.0)
    print(f"  previous best val acc: {best_acc:.4f}")
else:
    print("No checkpoint — starting fresh")

### Stage 1 — Train head only

In [ ]:
if not resume:
    freeze_backbone(model, classifier_attr)
    classifier = getattr(model, classifier_attr)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=LR_HEAD)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.1)

    for epoch in range(1, EPOCHS_STAGE1 + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion,
            desc=f"Stage1 Epoch {epoch}/{EPOCHS_STAGE1}",
        )
        val_loss, val_acc = validate(model, val_loader, criterion)
        scheduler.step(val_acc)
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            with open(STATE_PATH, "w") as f:
                json.dump({"best_val_acc": best_acc, "stage": 1}, f)
            print(f"  -> saved (best val acc: {best_acc:.4f})")
else:
    print("Stage 1 skipped (resuming from checkpoint)")

### Stage 2 — Full fine-tune

In [ ]:
unfreeze_all(model)
classifier = getattr(model, classifier_attr)
classifier_params_id = {id(p) for p in classifier.parameters()}
backbone_params = [p for p in model.parameters() if id(p) not in classifier_params_id]

optimizer = torch.optim.Adam([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": classifier.parameters(), "lr": LR_HEAD},
])
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.1)

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion,
        desc=f"Stage2 Epoch {epoch}/{EPOCHS_STAGE2}",
    )
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step(val_acc)
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        with open(STATE_PATH, "w") as f:
            json.dump({"best_val_acc": best_acc, "stage": 2}, f)
        print(f"  -> saved updated (best val acc: {best_acc:.4f})")

print(f"\nTraining complete. Best val acc: {best_acc:.4f}")

## Evaluation — Metrics & Confusion Matrix

In [ ]:
def get_class_names():
    builder = load_dataset_builder("taufiktrf/AQUA20")
    return builder.info.features["label"].names

@torch.no_grad()
def evaluate(model, loader):
    """Return true labels and the full softmax matrix.

    Keeping probabilities (not just argmax) is what makes confidence and
    near-miss analysis possible later — it costs ~130 KB.
    """
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        images, batch_labels = batch["image"].to(DEVICE), batch["label"]
        probs = torch.softmax(model(images), dim=1)
        all_probs.append(probs.cpu())
        all_labels.append(batch_labels)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()

class_names = get_class_names()
print(f"Classes ({len(class_names)}): {class_names}\n")

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
labels, probs = evaluate(model, test_loader)

# Everything below is derived from `probs`, so it can never disagree with it.
ranking = np.argsort(-probs, axis=1)          # class indices, best first
preds = ranking[:, 0]
top3_preds = ranking[:, :3]
rows = np.arange(len(labels))
confidence = probs[rows, ranking[:, 0]]       # p(top-1)
margin = confidence - probs[rows, ranking[:, 1]]   # p(top-1) - p(top-2)
true_prob = probs[rows, labels]               # p assigned to the correct class
correct = preds == labels

top1_acc = accuracy_score(labels, preds)
top3_acc = float(np.mean([labels[i] in top3_preds[i] for i in rows]))
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")

print("=" * 60)
print(f"Top-1 Accuracy:  {top1_acc:.4f} ({top1_acc * 100:.2f}%)")
print(f"Top-3 Accuracy:  {top3_acc:.4f} ({top3_acc * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}")
print("=" * 60)
print("\nPer-class Classification Report:")
print(classification_report(labels, preds, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix — {MODEL_NAME} (Top-1: {top1_acc*100:.2f}%)")
plt.tight_layout()
plt.savefig(CONFUSION_PATH, dpi=150)
plt.show()
print(f"Confusion matrix saved to {CONFUSION_PATH}")

## Error Analysis

Writes small CSV/JSON artifacts to `analysis/`. These download first and survive a truncated
pull, unlike the log they would otherwise only exist in.

In [ ]:
A = ANALYSIS_DIR
cm = confusion_matrix(labels, preds)

# ── 1. Per-class metrics, incl. top-3 recovery and confidence split ──────────
p_c, r_c, f_c, sup = precision_recall_fscore_support(
    labels, preds, labels=range(NUM_CLASSES), zero_division=0
)
per_class = []
for c in range(NUM_CLASSES):
    m = labels == c
    hit, miss = m & correct, m & ~correct
    per_class.append({
        "class": class_names[c],
        "support": int(sup[c]),
        "precision": round(float(p_c[c]), 4),
        "recall": round(float(r_c[c]), 4),
        "f1": round(float(f_c[c]), 4),
        # how often the right answer was at least in the top 3
        "top3_recall": round(float(np.mean([labels[i] in top3_preds[i]
                                            for i in np.flatnonzero(m)])), 4) if m.any() else 0.0,
        "mean_conf_correct": round(float(confidence[hit].mean()), 4) if hit.any() else None,
        "mean_conf_wrong": round(float(confidence[miss].mean()), 4) if miss.any() else None,
        # when wrong, how much probability did the true class still get?
        "mean_true_prob_when_wrong": round(float(true_prob[miss].mean()), 4) if miss.any() else None,
    })
per_class_df = pd.DataFrame(per_class).sort_values("f1")
per_class_df.to_csv(f"{A}/per_class.csv", index=False)

# ── 2. Confusion matrix as data (the PNG is not machine-readable) ────────────
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(f"{A}/confusion_matrix.csv")

# ── 3. Ranked confusion pairs ───────────────────────────────────────────────
pairs = [
    {"true": class_names[t], "predicted": class_names[p], "count": int(cm[t, p]),
     "pct_of_true_class": round(100.0 * cm[t, p] / max(cm[t].sum(), 1), 2)}
    for t in range(NUM_CLASSES) for p in range(NUM_CLASSES)
    if t != p and cm[t, p] > 0
]
pairs_df = pd.DataFrame(pairs).sort_values("count", ascending=False)
pairs_df.to_csv(f"{A}/confusion_pairs.csv", index=False)

# ── 4. Most confident mistakes — the most report-worthy failures ─────────────
wrong_idx = np.flatnonzero(~correct)
mistakes = pd.DataFrame({
    "test_index": wrong_idx,
    "true": [class_names[c] for c in labels[wrong_idx]],
    "predicted": [class_names[c] for c in preds[wrong_idx]],
    "confidence": np.round(confidence[wrong_idx], 4),
    "margin": np.round(margin[wrong_idx], 4),
    "true_class_prob": np.round(true_prob[wrong_idx], 4),
    "true_rank": [int(np.where(ranking[i] == labels[i])[0][0]) + 1 for i in wrong_idx],
}).sort_values("confidence", ascending=False)
mistakes.to_csv(f"{A}/confident_mistakes.csv", index=False)

# ── 5. Summary ──────────────────────────────────────────────────────────────
near_miss = int((mistakes["true_rank"] == 2).sum())
summary = {
    "model": MODEL_NAME,
    "n_test": int(len(labels)),
    "top1_accuracy": round(float(top1_acc), 4),
    "top3_accuracy": round(float(top3_acc), 4),
    "macro_precision": round(float(precision), 4),
    "macro_recall": round(float(recall), 4),
    "macro_f1": round(float(f1), 4),
    "weighted_f1": round(float(precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0)[2]), 4),
    "best_val_acc": round(float(best_acc), 4),
    "n_errors": int(len(wrong_idx)),
    "mean_confidence_correct": round(float(confidence[correct].mean()), 4),
    "mean_confidence_wrong": round(float(confidence[~correct].mean()), 4),
    "errors_where_true_was_rank2": near_miss,
    "errors_where_true_was_rank2_pct": round(100.0 * near_miss / max(len(wrong_idx), 1), 2),
    "epochs_stage1": EPOCHS_STAGE1,
    "epochs_stage2": EPOCHS_STAGE2,
    "weighted_sampler": True,
}
json.dump(summary, open(f"{A}/summary.json", "w"), indent=2)
np.save(f"{A}/probabilities.npy", probs.astype(np.float32))

# ── 6. Print the findings ───────────────────────────────────────────────────
print("=" * 68)
print("ERROR ANALYSIS")
print("=" * 68)
print(f"{len(wrong_idx)} errors out of {len(labels)} ({100*len(wrong_idx)/len(labels):.1f}%)\n")

print("Confidence — is the model wrong *confidently*?")
print(f"  mean confidence when correct : {summary['mean_confidence_correct']:.4f}")
print(f"  mean confidence when wrong   : {summary['mean_confidence_wrong']:.4f}")
print(f"  errors where the true class was still rank 2: "
      f"{near_miss} ({summary['errors_where_true_was_rank2_pct']:.1f}% of errors)\n")

print("5 weakest classes by F1:")
print(per_class_df.head(5).to_string(index=False), "\n")

print("10 most common confusions:")
print(pairs_df.head(10).to_string(index=False), "\n")

print("10 most confident mistakes:")
print(mistakes.head(10).to_string(index=False), "\n")

print(f"Artifacts written to {A}/:")
for f in sorted(os.listdir(A)):
    print(f"  {f}  ({os.path.getsize(os.path.join(A, f)) / 1024:.1f} KB)")

## Grad-CAM Explainability

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer, channels_last=False):
        self.model = model
        self.target_layer = target_layer
        self.channels_last = channels_last
        self.gradients = self.activations = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, m, inp, out): self.activations = out.detach()
    def _bwd_hook(self, m, gin, gout): self.gradients = gout[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()

        # Grad-CAM needs the backward pass, so autograd must stay enabled here.
        # requires_grad on the input guarantees a graph even if every parameter
        # is frozen.
        with torch.enable_grad():
            input_tensor = input_tensor.unsqueeze(0).requires_grad_(True)
            output = self.model(input_tensor)
            if class_idx is None:
                class_idx = output.argmax(dim=1).item()
            one_hot = torch.zeros_like(output)
            one_hot[0, class_idx] = 1
            self.model.zero_grad()
            output.backward(gradient=one_hot)

        with torch.no_grad():
            grads, acts = self.gradients, self.activations
            if self.channels_last:  # (N, H, W, C) -> (N, C, H, W)
                grads = grads.permute(0, 3, 1, 2)
                acts = acts.permute(0, 3, 1, 2)
            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam = (weights * acts).sum(dim=1)
            cam = torch.relu(cam).squeeze(0).cpu().numpy()

        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        h, w = input_tensor.shape[2:]
        cam = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
        return cam, class_idx

def unnormalize(tensor):
    img = tensor.cpu().clone()
    for t, m, s in zip(img, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    return img.permute(1, 2, 0).numpy()

def overlay_heatmap(img, cam, alpha=0.5):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return (alpha * heatmap + (1 - alpha) * 255 * img).astype(np.uint8)

In [ ]:
# How many Grad-CAM figures to produce, split between errors and successes.
# Misclassifications get the larger share — they are what the analysis is about.
GRADCAM_N_WRONG = 20
GRADCAM_N_RIGHT = 20


def select_gradcam_indices(labels, preds, confidence, n_wrong, n_right):
    """Pick a class-diverse sample of test indices, weighted toward errors.

    Within each class the most *confident* cases come first: a confident error
    is a far more interesting figure than one the model was already unsure of.

    labels/preds come from test_loader with shuffle=False, so position i in
    those arrays is index i in the test dataset.
    """
    def spread(pool, n):
        # Round-robin over true classes so one dominant class (fish, coral)
        # cannot swallow the whole quota.
        by_class = {}
        for idx in pool:
            by_class.setdefault(int(labels[idx]), []).append(int(idx))
        for c in by_class:
            by_class[c].sort(key=lambda i: -confidence[i])
        picked, classes = [], sorted(by_class)
        while len(picked) < n and any(by_class[c] for c in classes):
            for c in classes:
                if by_class[c] and len(picked) < n:
                    picked.append(by_class[c].pop(0))
        return picked

    wrong = spread(np.flatnonzero(labels != preds), n_wrong)
    right = spread(np.flatnonzero(labels == preds), n_right)
    return wrong, right


target_fn, channels_last = GRADCAM_TARGETS[MODEL_NAME]
target_layer = target_fn(model)
print(f"Target layer: {type(target_layer).__name__} (channels_last={channels_last})")
gradcam = GradCAM(model, target_layer, channels_last=channels_last)

wrong_sel, right_sel = select_gradcam_indices(
    labels, preds, confidence, GRADCAM_N_WRONG, GRADCAM_N_RIGHT
)
selected = wrong_sel + right_sel
n_classes = len({int(labels[i]) for i in selected})
print(f"Selected {len(wrong_sel)} misclassified + {len(right_sel)} correct "
      f"images, spanning {n_classes} classes")

# Grad-CAM runs one image at a time, so it needs its own batch_size=1 loader.
gradcam_loader = DataLoader(Subset(test_loader.dataset, selected), batch_size=1,
                            shuffle=False, num_workers=NUM_WORKERS)

manifest = []
for i, batch in enumerate(gradcam_loader):
    ds_idx = selected[i]
    label, pred = int(labels[ds_idx]), int(preds[ds_idx])
    img_tensor = batch["image"].to(DEVICE)

    # Explain the prediction that was actually recorded, not a fresh argmax.
    cam, _ = gradcam.generate(img_tensor.squeeze(0), class_idx=pred)
    img_np = np.clip(unnormalize(img_tensor.squeeze(0)), 0, 1)
    overlay = overlay_heatmap(img_np, cam)

    is_ok = label == pred
    conf = float(confidence[ds_idx])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np); axes[0].set_title(f"True: {class_names[label]}"); axes[0].axis("off")
    axes[1].imshow(cam, cmap="jet", vmin=0, vmax=1); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].axis("off")
    axes[2].set_title(f"Pred: {class_names[pred]} ({conf:.2f})" + ("" if is_ok else "  X"))
    plt.tight_layout()

    tag = "ok" if is_ok else "wrong"
    fname = f"{tag}_{i:03d}_{class_names[label]}-as-{class_names[pred]}.png"
    plt.savefig(f"{GRADCAM_DIR}/{fname}", dpi=150)
    plt.close(fig)
    manifest.append({"file": fname, "test_index": ds_idx, "true": class_names[label],
                     "predicted": class_names[pred], "confidence": round(conf, 4),
                     "correct": bool(is_ok)})
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(selected)}] saved")

# Manifest lives in analysis/ so it survives even if the images are truncated.
pd.DataFrame(manifest).to_csv(f"{ANALYSIS_DIR}/gradcam_manifest.csv", index=False)
print(f"Saved {len(selected)} Grad-CAM visualizations to {GRADCAM_DIR}/")
print(f"  misclassified -> {GRADCAM_DIR}/wrong_*.png")

## Outputs

In [ ]:
# /kaggle/working IS the kernel output — nothing to copy, just report what is there.
total = 0
print("Kernel output (/kaggle/working):")
for root, _, files in os.walk("."):
    if any(p in root for p in (".git", "__pycache__", ".ipynb_checkpoints")):
        continue
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        total += size
        print(f"  {os.path.relpath(path):58} {size/1e6:8.2f} MB")
print(f"\nTotal: {total/1e6:.1f} MB")
print("\nPull with:  kaggle kernels output farhantahsinkhan/<slug> -p <dir>")
print("Downloads run alphabetically and can truncate — 'analysis/' arrives first,")
print(f"'{CHECKPOINT_PATH}' last. If files look missing, re-pull before suspecting the run.")